[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch09-data-selection/04-du-lieu-re-va-chi-phi.ipynb)

# Dữ liệu rẻ hơn, và cái giá thật của nó: tăng cường, lặp lại dữ liệu, ROI

Hai sổ tay trước bỏ bớt dữ liệu. Sổ tay này đi theo chiều ngược lại: tạo thêm dữ liệu rẻ, dùng lại dữ liệu
đã có, rồi hỏi câu chương đặt ra ở cuối, là mọi kỹ thuật ấy có đáng tiền không. Phần đầu đo trên máy. Phần
sau tính lại các mô hình chi phí của chương và quét những giả định mà chúng dựa vào.

**Bạn sẽ làm:**
1. đo tăng cường dữ liệu (data augmentation) đúng và sai bất biến trên MNIST, rồi tính ví dụ phát hiện từ khoá;
2. đo khoảng cách miền khi dữ liệu "tổng hợp" khác dữ liệu thật, và các tỉ lệ pha trộn của bảng 9;
3. tính lại phép tính nhẩm 1.4 về lặp lại dữ liệu (data echoing), đo tỉ số R trên máy, và đo một batch lặp
   có giá trị bằng một batch mới không;
4. tính mức phạt truy cập ngẫu nhiên của bảng 13 và mô phỏng bộ đệm xáo trộn (shuffle buffer);
5. tính lại bảng 15, điểm hoà vốn của học chủ động và ROI khấu hao, rồi quét các giả định;
6. tính điểm hoà vốn của tiền huấn luyện một mô hình nền tảng;
7. đo vì sao giảm 99 phần trăm số phép toán không có nghĩa là giảm 99 phần trăm thời gian.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import warnings

import torch.nn.functional as F
import torchvision
from sklearn.datasets import load_digits

# Ma trận thưa CSR của PyTorch còn ở giai đoạn beta và in cảnh báo mỗi lần tạo; phép nhân vẫn đúng.
warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta state")

chu_so = torchvision.datasets.MNIST(DATA, train=True, download=True)
chu_so_kt = torchvision.datasets.MNIST(DATA, train=False, download=True)
A = chu_so.data.float().unsqueeze(1) / 255  # ảnh 1×28×28
nhan = chu_so.targets.clone()
A_kt = chu_so_kt.data[:5_000].float().unsqueeze(1) / 255
nhan_kt = chu_so_kt.targets[:5_000].clone()

thoi_trang = torchvision.datasets.FashionMNIST(DATA, train=True, download=True)
thoi_trang_kt = torchvision.datasets.FashionMNIST(DATA, train=False, download=True)
B = thoi_trang.data[:20_000].float().unsqueeze(1) / 255
nhan_b = thoi_trang.targets[:20_000].clone()
B_kt = thoi_trang_kt.data[:5_000].float().unsqueeze(1) / 255
nhan_b_kt = thoi_trang_kt.targets[:5_000].clone()
print(f"MNIST {len(A):,} ảnh, FashionMNIST {len(B):,} ảnh dùng ở đây; tập kiểm tra 5,000 ảnh mỗi bộ")


def mlp(an=256):
    return torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(784, an), torch.nn.ReLU(), torch.nn.Linear(an, 10))


def do_chinh_xac(m, X_, y_):
    with torch.no_grad():
        return 100 * (m(X_).argmax(1) == y_).float().mean().item()


def huan_luyen(X_, y_, so_buoc=1_500, batch=64, tang_cuong=None, seed=SEED):
    """MLP học `so_buoc` bước, mỗi bước một batch bốc ngẫu nhiên; tang_cuong(x, g) biến đổi batch."""
    dat_hat_giong(seed)
    g = torch.Generator().manual_seed(seed)
    m = mlp()
    toi_uu = torch.optim.Adam(m.parameters(), lr=1e-3, foreach=False)
    for _ in range(so_buoc):
        b = torch.randint(len(X_), (batch,), generator=g)
        xb = tang_cuong(X_[b], g) if tang_cuong else X_[b]
        mat_mat = F.cross_entropy(m(xb), y_[b])
        toi_uu.zero_grad()
        mat_mat.backward()
        toi_uu.step()
    return m

## 1. Tăng cường dữ liệu: đúng bất biến thì mới là dữ liệu rẻ

Tăng cường dữ liệu mở rộng tập bằng các phép biến đổi giữ nguyên nghĩa của nhãn. Chương nhấn mạnh rằng tăng
cường nên khớp với tính bất biến mà mô hình phải học khi triển khai. Chữ số là một ví dụ rõ: dịch vài pixel
hay xoay nhẹ không đổi chữ số, còn lật ngang thì có thể biến một chữ số thành một hình không phải chữ số nào.

Huấn luyện cùng một MLP, cùng 1,500 bước, trên 500 hoặc 2,000 ảnh thật, với ba cách tăng cường.

**Dự đoán:** 500 ảnh có tăng cường đúng có bằng được 2,000 ảnh không tăng cường không?

In [ ]:
def bien_doi(x, g, xoay=10, dich=2, lat=False):
    """Xoay ±xoay độ, dịch ±dich pixel, và lật ngang một nửa số ảnh nếu lat=True."""
    n = len(x)
    goc = (torch.rand(n, generator=g) * 2 - 1) * math.radians(xoay)
    tx = (torch.rand(n, generator=g) * 2 - 1) * (dich / 14)
    ty = (torch.rand(n, generator=g) * 2 - 1) * (dich / 14)
    huong = torch.where(torch.rand(n, generator=g) < 0.5, -1.0, 1.0) if lat else torch.ones(n)
    c, s = torch.cos(goc), torch.sin(goc)
    theta = torch.stack([torch.stack([c * huong, -s, tx], 1), torch.stack([s * huong, c, ty], 1)], 1)
    luoi = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, luoi, align_corners=False, padding_mode="zeros")


CACH = {
    "không tăng cường": None,
    "lật ngang": lambda x, g: bien_doi(x, g, xoay=0, dich=0, lat=True),
    "dịch 2 pixel + xoay 10 độ": lambda x, g: bien_doi(x, g),
}
chon = torch.randperm(len(A), generator=torch.Generator().manual_seed(SEED))
ket_qua_tc = {}
t0 = time.perf_counter()
for so_anh in (500, 2_000):
    for ten, tc in CACH.items():
        if so_anh == 2_000 and ten == "lật ngang":
            continue
        m = huan_luyen(A[chon[:so_anh]], nhan[chon[:so_anh]], tang_cuong=tc)
        ket_qua_tc[(so_anh, ten)] = do_chinh_xac(m, A_kt, nhan_kt)
        print(f"⏱ {so_anh:>5,} ảnh, {ten:<26} {ket_qua_tc[(so_anh, ten)]:.1f}% (đo trên máy này)")
do_tren_may("thời gian cả năm lần huấn luyện", time.perf_counter() - t0, "s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
nhom = [(k, v) for k, v in ket_qua_tc.items()]
mau = {"không tăng cường": "0.45", "lật ngang": "C3", "dịch 2 pixel + xoay 10 độ": "C2"}
ax1.barh(range(len(nhom)), [v for _, v in nhom], color=[mau[k[1]] for k, _ in nhom])
ax1.set_yticks(range(len(nhom)), [f"{k[0]:,} ảnh, {k[1]}" for k, _ in nhom], fontsize=9)
ax1.invert_yaxis()
for i, (_, v) in enumerate(nhom):
    ax1.text(v - 0.5, i, f"{v:.1f}%", va="center", ha="right", color="white", fontsize=9)
ax1.set_xlim(60, 100)
ax1.set_xlabel("độ chính xác trên tập kiểm tra MNIST (%)")
ax1.set_title("Cùng 1,500 bước huấn luyện (đo trên máy này)")

g = torch.Generator().manual_seed(1)
mau_anh = A[chon[:4]]
hang = [mau_anh, bien_doi(mau_anh, g, xoay=0, dich=0, lat=True), bien_doi(mau_anh, g)]
for r, (anh_, ten) in enumerate(zip(hang, ["gốc", "lật ngang", "dịch + xoay"])):
    for c in range(4):
        ax = ax2.inset_axes([c * 0.25, 1 - (r + 1) / 3, 0.23, 0.3])
        ax.imshow(anh_[c, 0], cmap="gray_r")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        if c == 0:
            ax.set_ylabel(ten, fontsize=8)
ax2.axis("off")
ax2.set_title("Cùng bốn ảnh, ba cách biến đổi")
plt.tight_layout()
plt.show()

Dịch và xoay nhẹ là đúng bất biến của chữ số, nên ở lần chạy này 500 ảnh có tăng cường vượt cả 2,000 ảnh
không tăng cường: phép biến đổi rẻ thay được cho dữ liệu thật đắt. Lật ngang thì làm hại, vì nó dạy mô hình
rằng một hình ảnh gương cũng mang nhãn chữ số ấy. Cùng một phép biến đổi, với quần áo thì vô hại: một chiếc
giày lật ngang vẫn là giày. Đó là lý do chương dặn chính sách tăng cường phải được kiểm định với mô hình đích
và bất biến khi triển khai.

Ví dụ 1.4 của chương đẩy ý này tới mức cực đoan: một hệ thống phát hiện từ khoá trên vi điều khiển cần hơn
10,000 bản ghi, nhưng quy trình năm tầng bắt đầu từ 500 bản ghi gốc.

In [ ]:
CAN = sach(10_000, nguon="256 KB · 10,000")
GOC_KWS = sach(500, nguon="20–50 nghìn đô la · 500")
GHI_THAP = sach(20, nguon="20–50 nghìn đô la") * 1_000
GHI_CAO = sach(50_000, nguon="20–50 nghìn đô la")
PHAN = sach(5, nguon="5 tầng · 5 phần trăm") / 100
print(f"mỗi bản ghi gốc phải thành {CAN / GOC_KWS:.0f} mẫu huấn luyện")
print(f"ghi âm toàn bộ: {GHI_THAP:,.0f}–{GHI_CAO:,.0f} đô la; ở {PHAN:.0%} mức ấy:"
      f" {PHAN * GHI_THAP:,.0f}–{PHAN * GHI_CAO:,.0f} đô la cho phần thu thập vật lý")

Chương ghi kèm điều kiện: độ chính xác mục tiêu và độ phủ âm học vẫn phải kiểm định. Phép tính chỉ nói về
tiền ghi âm, không nói 20 biến thể của một bản ghi có đáng giá 20 bản ghi thật không.

## 2. Khoảng cách miền, và tỉ lệ pha trộn

Dữ liệu tổng hợp có thể rất nhiều mà vẫn không khớp dữ liệu triển khai. Để đo điều đó mà không cần bộ mô
phỏng, sổ tay dùng một nguồn chữ số **khác miền**: 1,797 ảnh chữ số 8×8 của scikit-learn, phóng lên 28×28
và đặt vào giữa khung như MNIST. Cùng tác vụ, cùng mười lớp, nhưng nét, độ dày và độ phân giải khác. Coi
chúng là dữ liệu "tổng hợp" rẻ, còn MNIST là dữ liệu thật, đắt.

Bảng 9 của chương liệt kê bốn mức pha trộn: toàn tổng hợp, 80 tổng hợp với 20 thật, nửa nửa, và toàn thật.
Chương coi đó là khung kịch bản, không phải công thức. Giữ tổng số ảnh là 1,500 và đổi tỉ lệ.

**Dự đoán:** 20 phần trăm thật trộn 80 phần trăm tổng hợp tốt hơn hay kém hơn chỉ dùng phần thật ấy?

In [ ]:
digits = load_digits()
S = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16
S = F.pad(F.interpolate(S, size=(20, 20), mode="bilinear", align_corners=False), (4, 4, 4, 4))
nhan_s = torch.tensor(digits.target)
print(f"nguồn tổng hợp: {len(S):,} ảnh, {S.shape[-2]}×{S.shape[-1]}")

TONG = 1_500
PHAN_THAT = [1 - sach(100, nguon="100 · 80 + 20 · 50 + 50 · 100 phần trăm") / 100,  # toàn tổng hợp
             sach(20, nguon="80 + 20") / 100,
             sach(50, nguon="50 + 50") / 100,
             sach(100, nguon="50 + 50 · 100 phần trăm") / 100]
tron = torch.randperm(len(S), generator=torch.Generator().manual_seed(SEED))
ket_qua_tron, chi_that = [], {}
for p in PHAN_THAT:
    so_that = int(round(p * TONG))
    X_ = torch.cat([A[chon[:so_that]], S[tron[:TONG - so_that]]])
    y_ = torch.cat([nhan[chon[:so_that]], nhan_s[tron[:TONG - so_that]]])
    ket_qua_tron.append((p, so_that, do_chinh_xac(huan_luyen(X_, y_), A_kt, nhan_kt)))
    if 0 < so_that < TONG:
        chi_that[so_that] = do_chinh_xac(huan_luyen(A[chon[:so_that]], nhan[chon[:so_that]]), A_kt, nhan_kt)
for p, so_that, d in ket_qua_tron:
    them = f"; chỉ {so_that:,} ảnh thật: {chi_that[so_that]:.1f}%" if so_that in chi_that else ""
    print(f"⏱ {p:.0%} thật ({so_that:,} thật + {TONG - so_that:,} tổng hợp): {d:.1f}%{them} (đo trên máy này)")

# Khi ảnh thật cực hiếm: 50 ảnh thật, có và không có cả 1,797 ảnh tổng hợp.
it_that = do_chinh_xac(huan_luyen(A[chon[:50]], nhan[chon[:50]]), A_kt, nhan_kt)
it_that_tong_hop = do_chinh_xac(huan_luyen(torch.cat([A[chon[:50]], S]), torch.cat([nhan[chon[:50]], nhan_s])), A_kt, nhan_kt)
print(f"⏱ 50 ảnh thật: {it_that:.1f}%; 50 ảnh thật + {len(S):,} ảnh tổng hợp: {it_that_tong_hop:.1f}% (đo trên máy này)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot([100 * p for p, _, _ in ket_qua_tron], [d for *_, d in ket_qua_tron], "o-", color="C4", lw=2,
        label=f"pha trộn, tổng {TONG:,} ảnh")
ax.plot([100 * k / TONG for k in chi_that], list(chi_that.values()), "s", color="0.3", ms=8,
        label="chỉ phần ảnh thật, bỏ phần tổng hợp")
for p, so_that, d in ket_qua_tron:
    ax.annotate(f"{d:.1f}%", (100 * p, d), textcoords="offset points", xytext=(8, -12), fontsize=8)
ax.set_xticks([100 * p for p in PHAN_THAT], ["toàn tổng hợp", "20% thật", "50% thật", "toàn thật"])
ax.set_ylabel("độ chính xác trên MNIST thật (%)")
ax.set_ylim(40, 100)
ax.set_title("Bốn mức pha trộn của bảng 9 (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Toàn tổng hợp cho một mô hình gần như chỉ đoán được một nửa số chữ số thật: đó là khoảng cách miền, đo được.
Ở lần chạy này, khi đã có vài trăm ảnh thật, trộn thêm ảnh tổng hợp khác miền còn làm kém đi so với chỉ dùng
phần thật ấy, vì cùng số bước huấn luyện bị chia cho một phân phối khác. Chỉ khi ảnh thật cực hiếm, như 50
ảnh, nguồn tổng hợp mới thêm được chút tín hiệu, và chỉ chút ít: chưa tới hai điểm, trong khi khoảng cách tới
toàn thật vẫn hơn hai mươi điểm. Đó là điều chương nói khi gọi bảng 9 là khung kịch bản: tỉ lệ tốt
nhất tuỳ miền, độ trung thực của nguồn tổng hợp và quy trình kiểm định, và luôn phải kiểm định trên dữ liệu
triển khai thật.

## 3. Lặp lại dữ liệu khi quy trình dữ liệu là điểm nghẽn

Phép tính nhẩm 1.4 huấn luyện ResNet-50 trên ImageNet với RandAugment và MixUp. Quy trình dữ liệu trên CPU
đạt 300 ảnh mỗi giây, còn GPU huấn luyện được 800. Tỉ số $R = T_{\text{quy trình}} / T_{\text{GPU}}$ cho
biết GPU chờ bao lâu, và lặp lại mỗi batch $e$ lần nâng thông lượng hiệu dụng lên $\min(e \cdot 300, 800)$.

In [ ]:
QUY_TRINH = sach(300, nguon="300 · 800 ảnh mỗi giây")
GPU = sach(800, nguon="300 · 800 ảnh mỗi giây")
theo_sach("R = T_quy_trình / T_GPU", GPU / QUY_TRINH, sach=2.67, nguon="300 · 800 ảnh mỗi giây · ≈2.67", tol=0.005)
theo_sach("thời gian GPU chờ (%)", 100 * (1 - QUY_TRINH / GPU), sach=62.5, nguon="62.5 phần trăm")
theo_sach("mức sử dụng GPU, không lặp (%)", 100 * QUY_TRINH / GPU, sach=37.5, nguon="37.5 phần trăm")

SO_EPOCH = sach(90, nguon="90 epoch · 384,350 giây (106.8 giờ) · 37.5 phần trăm")
ANH_IN = sach(1.28e6, nguon="1.28 triệu ảnh")
T_SACH = sach(384_350, nguon="90 epoch · 384,350 giây (106.8 giờ)")
t_tinh = SO_EPOCH * ANH_IN / QUY_TRINH
print(f"90 × 1.28 triệu / 300 = {t_tinh:,.0f} giây; sách in {T_SACH:,} giây")
print(f"   {T_SACH:,} giây ứng với {T_SACH * QUY_TRINH / SO_EPOCH:,.0f} ảnh mỗi epoch: sách dùng số ảnh chưa làm tròn")
theo_sach("không lặp: giờ huấn luyện", T_SACH / 3600, sach=106.8, nguon="384,350 giây (106.8 giờ)")
print(f"   với đúng 1.28 triệu ảnh thì là {t_tinh / 3600:.2f} giờ")

E = sach(2, nguon="2 · 600 · 300 · 192,175 giây (53.4 giờ)")
theo_sach("lặp 2 lần: thông lượng hiệu dụng (ảnh/s)", min(E * QUY_TRINH, GPU), sach=600, nguon="2 · 600 · 300")
theo_sach("lặp 2 lần: giây huấn luyện", T_SACH / E, sach=192_175, nguon="192,175 giây (53.4 giờ)")
theo_sach("lặp 2 lần: giờ huấn luyện", T_SACH / E / 3600, sach=53.4, nguon="192,175 giây (53.4 giờ)")
print(f"Choi và cộng sự báo cáo thời gian thực tới mục tiêu giảm {sach(3.25, nguon='3.25×')}× trong một cấu hình mạng")

e_quet = np.arange(1, 6)
thong_luong = np.minimum(e_quet * QUY_TRINH, GPU)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(e_quet, thong_luong, "o-", color="C0", lw=2, label="thông lượng hiệu dụng")
ax.axhline(GPU, color="0.4", lw=1, ls="--")
ax.text(5, GPU + 20, "sức GPU, 800 ảnh/s", ha="right", fontsize=8, color="0.3")
ax.axvline(GPU / QUY_TRINH, color="C3", lw=1, ls=":")
ax.text(GPU / QUY_TRINH + 0.05, 220, "e = R ≈ 2.67", fontsize=8, color="C3")
for e_, tl in zip(e_quet, thong_luong):
    ax.annotate(f"{100 * tl / GPU:.0f}%", (e_, tl), textcoords="offset points", xytext=(0, -15), ha="center", fontsize=8)
ax.set_xlabel("hệ số lặp e")
ax.set_ylabel("ảnh mỗi giây GPU xử lý")
ax.set_ylim(0, 950)
ax.set_title("Lặp lại dữ liệu, ví dụ 1.4 (nhãn: mức sử dụng GPU)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Ba con số của ví dụ không khép từ chính những gì nó in: 90 nhân 1.28 triệu chia 300 là 384,000 giây, không
phải 384,350. Sách dùng số ảnh huấn luyện chưa làm tròn của ImageNet, và 106.8 giờ cũng chỉ đúng với số ấy.
Chênh lệch 0.1 phần trăm không đổi kết luận.

Trên đồ thị, lặp quá $e = R$ không nâng được mức sử dụng nữa, chỉ giảm độ đa dạng. Và khi $R < 1$ thì lặp
không đem lại gì. Câu hỏi đầu tiên vì vậy là: trên máy này, $R$ bằng bao nhiêu? Đo thời gian tăng cường một
batch trên CPU so với thời gian một bước huấn luyện.

In [ ]:
mau_b = B[:64]
g = torch.Generator().manual_seed(0)
m = mlp()
toi_uu = torch.optim.Adam(m.parameters(), lr=1e-3, foreach=False)
for _ in range(5):  # khởi động
    bien_doi(mau_b, g, lat=True)
    F.cross_entropy(m(mau_b), nhan_b[:64]).backward()
LAP = 100
t0 = time.perf_counter()
for _ in range(LAP):
    bien_doi(mau_b, g, lat=True)
t_quy_trinh = (time.perf_counter() - t0) / LAP
t0 = time.perf_counter()
for _ in range(LAP):
    mat_mat = F.cross_entropy(m(mau_b), nhan_b[:64])
    toi_uu.zero_grad()
    mat_mat.backward()
    toi_uu.step()
t_buoc = (time.perf_counter() - t0) / LAP
do_tren_may("tăng cường một batch 64 ảnh", 1e3 * t_quy_trinh, "ms")
do_tren_may("một bước huấn luyện trên batch ấy", 1e3 * t_buoc, "ms")
do_tren_may("R trên máy này", t_quy_trinh / t_buoc)
print("R < 1: lặp lại dữ liệu không giúp gì ở đây" if t_quy_trinh < t_buoc else "R > 1: lặp lại dữ liệu có thể giúp")

Một MLP nhỏ với phép tăng cường nhẹ thì quy trình dữ liệu không phải điểm nghẽn, nên ở đây lặp lại dữ liệu
chẳng cứu được giây nào. Nhưng câu hỏi thứ hai vẫn đo được trên máy nào cũng như nhau: **một batch lặp có giá
trị bằng một batch mới không?** Ví dụ 1.4 giả định là có. Mô phỏng một quy trình chỉ đọc được 3,000 ảnh
FashionMNIST mới, và dùng mỗi batch $e$ lần. Lặp trước bước tăng cường cho mỗi lần một biến thể mới, còn lặp
sau bước tăng cường thì dùng lại đúng tensor cũ. Mốc so sánh là đọc được $e$ lần nhiều ảnh mới hơn.

In [ ]:
def lap_lai(so_moi, e, cho_lap, seed):
    """Quy trình đọc `so_moi` ảnh mới đúng một lần; mỗi batch được huấn luyện e lần."""
    dat_hat_giong(seed)
    g = torch.Generator().manual_seed(seed + 100)
    m = mlp()
    toi_uu = torch.optim.Adam(m.parameters(), lr=1e-3, foreach=False)
    thu_tu = torch.randperm(len(B), generator=g)[:so_moi]
    for i in range(0, so_moi, 64):
        b = thu_tu[i:i + 64]
        da_tang = bien_doi(B[b], g, lat=True)
        for k in range(e):
            xb = da_tang if (cho_lap == "sau tăng cường" or k == 0) else bien_doi(B[b], g, lat=True)
            mat_mat = F.cross_entropy(m(xb), nhan_b[b])
            toi_uu.zero_grad()
            mat_mat.backward()
            toi_uu.step()
    return do_chinh_xac(m, B_kt, nhan_b_kt)


MOI = 3_000
CAU_HINH = [("không lặp", MOI, 1, "trước tăng cường"),
            ("lặp 2, trước tăng cường", MOI, 2, "trước tăng cường"), ("lặp 2, sau tăng cường", MOI, 2, "sau tăng cường"),
            ("ảnh mới, gấp 2", 2 * MOI, 1, "trước tăng cường"),
            ("lặp 4, trước tăng cường", MOI, 4, "trước tăng cường"), ("lặp 4, sau tăng cường", MOI, 4, "sau tăng cường"),
            ("ảnh mới, gấp 4", 4 * MOI, 1, "trước tăng cường")]
gia_tri_lap = {}
for ten, so_moi, e, cho in CAU_HINH:
    lan = [lap_lai(so_moi, e, cho, s) for s in (SEED, SEED + 1, SEED + 2)]
    gia_tri_lap[ten] = (e * so_moi // MOI if "mới" not in ten else so_moi // MOI, np.mean(lan), np.std(lan))
    print(f"⏱ {ten:<24} {np.mean(lan):.1f} ± {np.std(lan):.1f}% (3 hạt giống, đo trên máy này)")
khong_lap = gia_tri_lap["không lặp"][1]
for e in (2, 4):
    loi_moi = gia_tri_lap[f"ảnh mới, gấp {e}"][1] - khong_lap
    for cho in ("trước", "sau"):
        loi_lap = gia_tri_lap[f"lặp {e}, {cho} tăng cường"][1] - khong_lap
        print(f"⏱ lặp {e}, {cho} tăng cường: lấy lại {100 * loi_lap / loi_moi:.0f}% phần lợi của ảnh mới gấp {e}"
              " (đo trên máy này)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
nhom_mau = {"không lặp": "0.45", "trước": "C0", "sau": "C1", "mới": "C2"}
for i, (ten, (buoc, tb, ds)) in enumerate(gia_tri_lap.items()):
    loai = "không lặp" if ten == "không lặp" else ("mới" if "mới" in ten else ("trước" if "trước" in ten else "sau"))
    ax.bar(i, tb, yerr=ds, color=nhom_mau[loai], capsize=3)
    ax.text(i, tb + ds + 0.6, f"{tb:.1f}", ha="center", fontsize=8)
ax.set_xticks(range(len(gia_tri_lap)), [t.replace(", ", "\n") for t in gia_tri_lap], fontsize=8)
ax.set_ylim(50, max(tb for _, tb, _ in gia_tri_lap.values()) + 6)
ax.set_ylabel("độ chính xác FashionMNIST (%)")
ax.set_title("Một batch lặp có đáng một batch mới? (đo trên máy này)")
plt.tight_layout()
plt.show()

Lặp có lợi thật: ở cả hai hệ số, mọi cách lặp đều hơn không lặp. Nhưng một batch lặp **không** đáng một batch
mới. Lặp 2 lần chỉ lấy lại khoảng một nửa phần lợi của việc đọc gấp đôi ảnh mới, lặp 4 lần khoảng hai phần
ba phần lợi của gấp bốn, và ảnh mới vẫn dẫn hai tới ba điểm. Ví dụ 1.4 ghi rõ 53.4 giờ chỉ đúng nếu dữ liệu
lặp có giá trị ngang dữ liệu mới. Ở đây nó không ngang, nên để tới cùng độ chính xác sẽ cần nhiều bước hơn.

Lặp **trước** hay **sau** bước tăng cường thì ở đây không phân biệt được: chênh lệch nhỏ hơn độ lệch chuẩn giữa
ba hạt giống, có lẽ vì phép tăng cường ở đây nhẹ. Đó là lý do chương dặn hệ số lặp hữu ích phải đo cho từng
tải, vì nó tuỳ điểm chèn, tăng cường, xáo trộn và chỉ số mục tiêu. Kết quả này là của một MLP nhỏ trên
FashionMNIST, không phải của ResNet-50.

## 4. Cái giá của truy cập ngẫu nhiên

Chọn lọc theo chỉ số biến những lần đọc tuần tự lớn thành nhiều lần đọc nhỏ rải rác. Bảng 13 minh hoạ lần
đọc ngẫu nhiên 4 KB trên một thiết bị: thông lượng ngẫu nhiên bằng số lần đọc mỗi giây (IOPS) nhân 4 KB.

In [ ]:
KB = sach(4, nguon="4 KB") / 1_000  # MB mỗi lần đọc, theo đơn vị thập phân của bảng
THIET_BI = [  # tên, MB/s tuần tự, IOPS, mức phạt của sách
    ("HDD", sach(150, nguon="~150 MB/s", tol=0.5), sach(100, nguon="~100 IOPS", tol=0.5)),
    ("SSD SATA", sach(550, nguon="~550 · ~10,000 IOPS", tol=0.5), sach(10_000, nguon="~10,000 IOPS", tol=0.5)),
    ("NVMe", sach(3_500, nguon="~3,500 · ~500,000 IOPS", tol=0.5), sach(500_000, nguon="~500,000 IOPS", tol=0.5)),
]
ngau_nhien = {ten: iops * KB for ten, _, iops in THIET_BI}
theo_sach("HDD: MB/s ngẫu nhiên", ngau_nhien["HDD"], sach=0.4, nguon="~0.4 MB/s", tol=1e-9)
theo_sach("SSD SATA: MB/s ngẫu nhiên", ngau_nhien["SSD SATA"], sach=40, nguon="~10,000 IOPS · ~40", tol=1e-9)
theo_sach("NVMe: MB/s ngẫu nhiên", ngau_nhien["NVMe"], sach=2_000, nguon="~500,000 IOPS · ~2,000", tol=1e-9)
theo_sach("HDD: chậm hơn", THIET_BI[0][1] / ngau_nhien["HDD"], sach=375, nguon="375×")
# 550 / 40 = 13.75, sách làm tròn lên 13.8: đúng nửa đơn vị cuối, nên cần nới sai số một chút.
theo_sach("SSD SATA: chậm hơn", THIET_BI[1][1] / ngau_nhien["SSD SATA"], sach=13.8, nguon="~40 · 13.8×", tol=0.051)
theo_sach("NVMe: chậm hơn", THIET_BI[2][1] / ngau_nhien["NVMe"], sach=1.75, nguon="~2,000 · 1.75×", tol=1e-9)

Mức phạt lớn nhất rơi vào ổ đĩa quay, và nhỏ dần khi thiết bị phục vụ được nhiều lần đọc song song. Cách
chương đưa ra để giữ đọc tuần tự mà vẫn có lợi ích thống kê của lấy mẫu ngẫu nhiên là **bộ đệm xáo trộn**:
đọc shard tuần tự vào bộ nhớ rồi bốc ngẫu nhiên bên trong.

Mô phỏng một trường hợp xấu: 10,000 ảnh FashionMNIST được ghi xuống đĩa theo thứ tự lớp, mỗi lớp một đoạn
liền khoảng 1,000 ảnh. Bộ đệm giữ $K$ ảnh; mỗi bước nó trả ra một ảnh bốc ngẫu nhiên trong bộ đệm và đọc tuần
tự ảnh kế tiếp vào chỗ trống. Đo một batch 64 ảnh chứa trung bình bao nhiêu lớp.

In [ ]:
N_XT = 10_000
theo_lop = torch.argsort(nhan_b[:N_XT] * N_XT + torch.arange(N_XT))  # thứ tự trên đĩa: xếp theo lớp


def bo_dem_xao_tron(K, seed=SEED):
    rng = np.random.default_rng(seed)
    dong = theo_lop.tolist()
    dem = dong[:K]
    ra = []
    for tiep in dong[K:] + [None] * K:
        j = rng.integers(len(dem))
        ra.append(dem[j])
        if tiep is None:
            dem[j] = dem[-1]
            dem.pop()
        else:
            dem[j] = tiep
    return np.array(ra)


CO_DEM = [1, 64, 256, 1_000, 4_000, N_XT]
lop_moi_batch = []
for K in CO_DEM:
    ra = bo_dem_xao_tron(K)
    assert sorted(ra.tolist()) == list(range(N_XT))  # mỗi ảnh ra đúng một lần
    lop = nhan_b[torch.tensor(ra)].numpy()
    so_lop = np.mean([len(set(lop[i:i + 64])) for i in range(0, N_XT - 63, 64)])
    lop_moi_batch.append(so_lop)
    print(f"bộ đệm {K:>6,} ảnh ({K * 784 / 1e6:6.2f} MB): trung bình {so_lop:.1f} lớp mỗi batch 64 ảnh")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.semilogx(CO_DEM, lop_moi_batch, "o-", color="C0", lw=2)
ax.axhline(10, color="0.5", lw=0.8, ls="--")
ax.text(1.2, 9.4, "đủ 10 lớp", fontsize=8, color="0.35")
ax.axvline(1_000, color="C3", lw=0.8, ls=":")
ax.text(1_050, 2, "một lớp liền ≈ 1,000 ảnh", fontsize=8, color="C3")
ax.set_xticks(CO_DEM, [f"{k:,}" for k in CO_DEM])
ax.minorticks_off()
ax.set_xlabel("cỡ bộ đệm xáo trộn K (ảnh, thang log)")
ax.set_ylabel("số lớp trung bình trong một batch")
ax.set_ylim(0, 11)
ax.set_title("Bộ đệm phải lớn gấp nhiều lần đoạn liền trên đĩa (mô phỏng)")
plt.tight_layout()
plt.show()

Bộ đệm nhỏ thì batch gần như chỉ có một lớp: đọc tuần tự rất nhanh, nhưng gradient mỗi bước chỉ thấy một
lớp. Bộ đệm bằng đúng một đoạn liền (1,000 ảnh) vẫn chỉ trộn được chừng bốn lớp, vì nó chủ yếu chứa vài
lớp vừa đọc gần nhất. Phải lớn gấp nhiều lần đoạn liền mà dữ liệu được ghi trên đĩa thì batch
mới trộn đủ, và ở đây chỉ bộ đệm chứa cả tập mới cho đủ mười lớp. Đó là lý do chương
dặn cân bằng các shard **trước** khi phân phối: khi mỗi worker giữ bộ đệm riêng trên một shard, việc xáo trộn
chỉ mang tính cục bộ, và một lớp hiếm chỉ có trong vài shard sẽ được phủ không đều.

## 5. Mô hình chi phí: dữ liệu chiếm bao nhiêu?

Tổng chi phí dữ liệu gồm bốn thành phần cộng lại: thu thập, gán nhãn, lưu trữ và xử lý. Phép tính nhẩm 1.5
định giá một kịch bản cỡ ImageNet, và chương nói rõ tỉ lệ thu được là hệ quả của giả định.

In [ ]:
ANH = sach(1.2e6, nguon="1.2 triệu ảnh · 50,000 · 0.05 · 60,000")
GIAY_PHEP = sach(50_000, nguon="1.2 triệu ảnh · 50,000")
GIA_NHAN = sach(0.05, nguon="50,000 · 0.05 · 60,000")
GB = sach(150, nguon="150 GB · 0.02 · 12 tháng · 36")
GIA_GB = sach(0.02, nguon="150 GB · 0.02")
THANG = sach(12, nguon="0.02 · 12 tháng")
LAN_CHAY = sach(30, nguon="30 lần chạy · 100 epoch · 24 giờ · 8 A100")
GIO_MOI_LAN = sach(24, nguon="100 epoch · 24 giờ · 8 A100")
SO_GPU = sach(8, nguon="24 giờ · 8 A100")
GIA_GIO = sach(4, nguon="5,760 giờ GPU · 4 đô la · 23,040")


def chi_phi(gia_nhan=GIA_NHAN, lan_chay=LAN_CHAY, giu=1.0):
    """Bốn thành phần của bảng 15; `giu` là phần dữ liệu giữ lại, chọn trước khi gán nhãn."""
    return {"giấy phép dữ liệu thô": GIAY_PHEP,
            "nhãn": giu * ANH * gia_nhan,
            "lưu trữ": giu * GB * GIA_GB * THANG,
            "tính toán": giu * lan_chay * GIO_MOI_LAN * SO_GPU * GIA_GIO}


phan = chi_phi()
theo_sach("nhãn (đô la)", phan["nhãn"], sach=60_000, nguon="0.05 · 60,000")
theo_sach("lưu trữ (đô la)", phan["lưu trữ"], sach=36, nguon="12 tháng · 36")
theo_sach("giờ GPU của chiến dịch", LAN_CHAY * GIO_MOI_LAN * SO_GPU, sach=5_760, nguon="5,760 giờ GPU")
theo_sach("tính toán (đô la)", phan["tính toán"], sach=23_040, nguon="4 đô la · 23,040")
tong = sum(phan.values())
theo_sach("tổng (đô la)", tong, sach=133_076, nguon="133,076 · 82.7 · 17.3 phần trăm")
theo_sach("phần dữ liệu (%)", 100 * (tong - phan["tính toán"]) / tong, sach=82.7, nguon="133,076 · 82.7")
theo_sach("phần tính toán (%)", 100 * phan["tính toán"] / tong, sach=17.3, nguon="82.7 · 17.3 phần trăm")
print(f"chỉ tính một lần chạy: tính toán {phan['tính toán'] / LAN_CHAY:,.0f} đô la, thấp hơn {LAN_CHAY} lần")

# Câu hỏi mở đầu của mô hình chi phí: thêm 10,000 nhãn hay thêm giờ GPU?
THEM = sach(10_000, nguon="10,000 mẫu")
print(f"{THEM:,} nhãn đám đông = {THEM * GIA_NHAN:,.0f} đô la = {THEM * GIA_NHAN / GIA_GIO:,.0f} giờ GPU ở giá của bảng 15")

Ở các giả định này, dữ liệu chiếm hơn bốn phần năm. Nhưng chương liệt kê cả khoảng giá: nhãn từ 0.01 đô la
với đám đông tới 200 đô la với chuyên gia. Hãy quét ba giả định của bảng 15: giá nhãn, số lần chạy của chiến
dịch, và phần dữ liệu giữ lại sau chọn lọc.

In [ ]:
NHAN_RE = sach(0.01, nguon="0.001–10 đô la · 0.01–200 đô la")
NHAN_DAT = sach(200, nguon="0.01–200 đô la")
gia = np.logspace(np.log10(NHAN_RE), np.log10(NHAN_DAT), 60)
lan = np.arange(1, 301)
giu = np.linspace(0.05, 1, 40)


def phan_du_lieu(p):
    return 100 * (1 - p["tính toán"] / sum(p.values()))


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 3.8))
ax1.semilogx(gia, [phan_du_lieu(chi_phi(gia_nhan=x)) for x in gia], color="C0", lw=2)
ax1.plot(GIA_NHAN, phan_du_lieu(phan), "o", color="C3")
ax1.annotate("bảng 15: 82.7%", (GIA_NHAN, phan_du_lieu(phan)), textcoords="offset points", xytext=(8, -14), fontsize=8)
ax1.set_xlabel("giá mỗi nhãn (đô la, thang log)")
ax1.set_ylabel("phần dữ liệu trong tổng chi phí (%)")
ax1.set_ylim(0, 100)
ax1.set_title("Giá nhãn")

ax2.semilogx(lan, [phan_du_lieu(chi_phi(lan_chay=k)) for k in lan], color="C0", lw=2)
ax2.plot(LAN_CHAY, phan_du_lieu(phan), "o", color="C3")
ax2.annotate("30 lần chạy", (LAN_CHAY, phan_du_lieu(phan)), textcoords="offset points", xytext=(6, 6), fontsize=8)
ax2.plot(1, phan_du_lieu(chi_phi(lan_chay=1)), "o", color="0.3")
ax2.annotate("1 lần chạy", (1, phan_du_lieu(chi_phi(lan_chay=1))), textcoords="offset points", xytext=(6, -12), fontsize=8)
ax2.set_xlabel("số lần chạy của chiến dịch (thang log)")
ax2.set_ylim(0, 100)
ax2.set_title("Số lần chạy")

ax3.plot(100 * giu, [sum(chi_phi(giu=f).values()) / 1_000 for f in giu], color="C2", lw=2, label="tổng")
ax3.plot(100 * giu, [GIAY_PHEP / 1_000] * len(giu), color="0.4", lw=1, ls="--", label="giấy phép, trọn gói")
ax3.set_xlabel("phần dữ liệu giữ lại, chọn trước khi gán nhãn (%)")
ax3.set_ylabel("nghìn đô la")
ax3.set_ylim(0, 150)
ax3.set_title("Chọn lọc chỉ cắt phần tỉ lệ theo mẫu")
ax3.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

nua = chi_phi(giu=0.5)
print(f"giữ 50%: tổng {sum(nua.values()):,.0f} đô la, tức {sum(nua.values()) / tong:.0%} của {tong:,.0f}")

Phần dữ liệu đổi từ thiểu số thành gần như tất cả chỉ bằng cách kéo giá nhãn trong khoảng chương nêu, và một
chiến dịch vài trăm lần chạy thì ngược lại. Chương vì vậy nói một phép tính ROI đầy đủ phải nêu số lần chạy
ra và định giá cả hai nhóm, thay vì đoán nhóm nào chi phối. Biểu đồ thứ ba thêm một điều: phí giấy phép
trọn gói không co lại khi chọn lọc, nên giữ một nửa dữ liệu không cắt được một nửa tổng chi phí.

## 6. Điểm hoà vốn, và ROI khấu hao

Chương định nghĩa ROI bằng phần tiết kiệm trừ khoản đầu tư, chia cho khoản đầu tư. Phân tích hoà vốn của
chương giả sử mỗi nhãn 10 đô la, học chủ động bắt đầu từ 1,000 nhãn và mỗi vòng gửi 100 truy vấn với chi phí
suy luận 50 đô la. Mốc ngẫu nhiên cần 5,000 nhãn để đạt mục tiêu, học chủ động cần 2,000.

In [ ]:
GIA_10 = sach(10, nguon="10 đô la · 1,000 · 10,000")
BAT_DAU = sach(1_000, nguon="10 đô la · 1,000")
MOI_VONG = sach(100, nguon="100 truy vấn · 50 đô la")
SUY_LUAN = sach(50, nguon="100 truy vấn · 50 đô la")
NGAU_NHIEN = sach(5_000, nguon="50 đô la · 5,000 · 2,000")
CHU_DONG = sach(2_000, nguon="5,000 · 2,000 · 50,000")
theo_sach("mốc ngẫu nhiên (đô la)", NGAU_NHIEN * GIA_10, sach=50_000, nguon="2,000 · 50,000 · 10 vòng")
so_vong = (CHU_DONG - BAT_DAU) / MOI_VONG
theo_sach("số vòng truy vấn", so_vong, sach=10, nguon="50,000 · 10 vòng")


def chi_phi_al(so_nhan, moi_vong_ton=SUY_LUAN):
    return so_nhan * GIA_10 + (so_nhan - BAT_DAU) / MOI_VONG * moi_vong_ton


theo_sach("học chủ động (đô la)", chi_phi_al(CHU_DONG), sach=20_500, nguon="10 vòng · 20,500")


def roi(tiet_kiem_goc, dau_tu):
    return 100 * (tiet_kiem_goc - dau_tu) / dau_tu


theo_sach("ROI (%)", roi(NGAU_NHIEN * GIA_10, chi_phi_al(CHU_DONG)), sach=143.9, nguon="20,500 · 143.9 phần trăm")

# Quét: mỗi vòng tốn bao nhiêu (chấm cả kho, huấn luyện lại) thì ROI về 0?
GIAM_20 = sach(20, nguon="20 phần trăm") / 100
muc_giam = [GIAM_20, 0.4, 1 - CHU_DONG / NGAU_NHIEN]
ton_vong = np.logspace(0, 4.5, 80)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for gm, mau_ in zip(muc_giam, ("C3", "C1", "C2")):
    can = NGAU_NHIEN * (1 - gm)
    ax.semilogx(ton_vong, [roi(NGAU_NHIEN * GIA_10, chi_phi_al(can, c)) for c in ton_vong], color=mau_, lw=2,
                label=f"bớt {gm:.0%} số nhãn ({can:,.0f} nhãn, {(can - BAT_DAU) / MOI_VONG:.0f} vòng)")
    hoa_von = (NGAU_NHIEN - can) * GIA_10 / ((can - BAT_DAU) / MOI_VONG)
    print(f"bớt {gm:.0%} số nhãn: hoà vốn khi mỗi vòng tốn {hoa_von:,.0f} đô la")
ax.axhline(0, color="0.3", lw=1)
ax.plot(SUY_LUAN, roi(NGAU_NHIEN * GIA_10, chi_phi_al(CHU_DONG)), "o", color="0.1")
ax.annotate("ví dụ của chương: 143.9%", (SUY_LUAN, roi(NGAU_NHIEN * GIA_10, chi_phi_al(CHU_DONG))),
            textcoords="offset points", xytext=(8, 4), fontsize=8)
ax.set_xlabel("chi phí mỗi vòng truy vấn (đô la, thang log)")
ax.set_ylabel("ROI (%)")
ax.set_ylim(-100, 320)
ax.set_title("ROI của học chủ động theo chi phí mỗi vòng")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Bớt 20 phần trăm số nhãn thì 30 vòng truy vấn phải trả, và chỉ cần mỗi vòng tốn vài trăm đô la cho chấm điểm
và huấn luyện lại là ROI âm. Đó là câu cảnh báo của chương: giảm 20 phần trăm số nhãn vẫn có thể cho ROI âm
nếu chấm điểm và huấn luyện lại đắt.

### ROI khấu hao

Một khoản đầu tư hạ tầng thì nên đánh giá qua nhiều lần chạy. Bảng 17: ống loại trùng tốn 50,000 đô la công
kỹ thuật, tính chữ ký MinHash một lần thêm 5,000, và mỗi lần chạy sau đó tiết kiệm 10,000.

In [ ]:
XAY = sach(50_000, nguon="50,000 · 5,000 · 10,000 mỗi lần chạy")
CHU_KY = sach(5_000, nguon="50,000 · 5,000")
MOI_LAN = sach(10_000, nguon="10,000 mỗi lần chạy")
theo_sach("đầu tư một lần (đô la)", XAY + CHU_KY, sach=55_000, nguon="55,000")


def roi_khau_hao(n):
    return 100 * (n * MOI_LAN - (XAY + CHU_KY)) / (XAY + CHU_KY)


theo_sach("1 lần chạy: lỗ (%)", -roi_khau_hao(sach(1, nguon="ở 1 · 5 · 10 · 50 lần chạy")), sach=81.8, nguon="âm 81.8")
theo_sach("5 lần chạy: lỗ (%)", -roi_khau_hao(sach(5, nguon="ở 1 · 5 · 10 · 50 lần chạy")), sach=9.1, nguon="âm 9.1")
theo_sach("10 lần chạy: ROI (%)", roi_khau_hao(sach(10, nguon="ở 1 · 5 · 10 · 50 lần chạy")), sach=81.8, nguon="âm 9.1 · 81.8")
theo_sach("50 lần chạy: ROI (%)", roi_khau_hao(sach(50, nguon="ở 1 · 5 · 10 · 50 lần chạy")), sach=809.1, nguon="809.1 phần trăm")
hoa_von_n = (XAY + CHU_KY) / MOI_LAN
print(f"hoà vốn ở {hoa_von_n:.1f} lần chạy (sổ tay tính, chương không in)")

n_quet = np.arange(1, 61)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(n_quet, roi_khau_hao(n_quet), color="C0", lw=2)
ax.axhline(0, color="0.3", lw=1)
for n_ in (1, 5, 10, 50):
    ax.plot(n_, roi_khau_hao(n_), "o", color="C3")
    ax.annotate(f"{n_} lần: {roi_khau_hao(n_):.1f}%", (n_, roi_khau_hao(n_)), textcoords="offset points",
                xytext=(10, -4) if n_ == 1 else (-8, 8), ha="left" if n_ == 1 else "right", fontsize=8)
ax.vlines(hoa_von_n, 0, 420, color="C2", lw=0.8, ls=":")
ax.text(hoa_von_n + 0.6, 450, f"hoà vốn ≈ {hoa_von_n:.1f} lần", fontsize=8, color="C2")
ax.set_xlabel("số lần chạy dùng lại ống loại trùng")
ax.set_ylabel("ROI khấu hao (%)")
ax.set_title("Bảng 18: lỗ ở lần đầu, lãi khi dùng lại")
plt.tight_layout()
plt.show()

ROI khấu hao tăng tuyến tính theo số lần dùng lại, và đổi dấu giữa lần chạy thứ năm và thứ sáu. Cùng một khoản
đầu tư là lỗ nặng cho một dự án làm một lần và lãi lớn cho một kho dùng chung nhiều năm. Với một lần chạy duy
nhất, chương nói cách đơn giản như lấy mẫu ngẫu nhiên hay tăng cường cơ bản có thể cho ROI tốt hơn.

## 7. Mô hình nền tảng: khấu hao tiền huấn luyện

Học tự giám sát (self-supervised learning) gỡ việc gán nhãn riêng của tác vụ khỏi tiền huấn luyện. Phần tiết
kiệm đến từ khấu hao: tiền huấn luyện đắt làm một lần, rồi dùng lại cho nhiều tác vụ. Kịch bản của chương:
một công ty xây 10 bộ phân loại.

In [ ]:
TAC_VU = sach(10, nguon="10 bộ phân loại · 100,000 nhãn")
NHAN_TU_DAU = sach(100_000, nguon="10 bộ phân loại · 100,000 nhãn · 1 đô la")
GIA_1 = sach(1, nguon="100,000 nhãn · 1 đô la · 1 triệu đô la")
GIO_TU_DAU = sach(10_000, nguon="1 triệu đô la · 10,000 giờ GPU")
TIEN_HL = sach(10_000, nguon="10,000 giờ GPU một lần · 1,000 nhãn")
NHAN_TINH_CHINH = sach(1_000, nguon="10,000 giờ GPU một lần · 1,000 nhãn · 50 giờ GPU")
GIO_TINH_CHINH = sach(50, nguon="1,000 nhãn · 50 giờ GPU · 10 nghìn đô la")

theo_sach("từ đầu: tiền nhãn (đô la)", TAC_VU * NHAN_TU_DAU * GIA_1, sach=1e6, nguon="1 đô la · 1 triệu đô la")
gio_moi_tac_vu = GIO_TU_DAU / TAC_VU
theo_sach("từ đầu: giờ GPU mỗi tác vụ", gio_moi_tac_vu, sach=1_000, nguon="1,000 giờ GPU mỗi tác vụ")
theo_sach("tinh chỉnh: tiền nhãn (đô la)", TAC_VU * NHAN_TINH_CHINH * GIA_1, sach=10_000, nguon="50 giờ GPU · 10 nghìn đô la")
theo_sach("tinh chỉnh: tổng giờ GPU ở 10 tác vụ", TIEN_HL + TAC_VU * GIO_TINH_CHINH, sach=10_500,
          nguon="10,000 so với 10,500")
theo_sach("tiền nhãn giảm", NHAN_TU_DAU / NHAN_TINH_CHINH, sach=100, nguon="100× · 20×")
theo_sach("tính toán biên mỗi tác vụ giảm", gio_moi_tac_vu / GIO_TINH_CHINH, sach=20, nguon="100× · 20×")
theo_sach("tinh chỉnh so với một lần huấn luyện đầy đủ (%)", 100 * GIO_TINH_CHINH / gio_moi_tac_vu, sach=5,
          nguon="100 phần trăm · 1–5 phần trăm")
diem_cat = TIEN_HL / (gio_moi_tac_vu - GIO_TINH_CHINH)
theo_sach("điểm hoà vốn (tác vụ)", diem_cat, sach=10.5, nguon="10,000 so với 10,500 · ≈10.5 tác vụ", tol=0.05)

n_tv = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(n_tv, n_tv * gio_moi_tac_vu / 1_000, color="0.35", lw=2, label="huấn luyện từ đầu")
ax.plot(n_tv, (TIEN_HL + n_tv * GIO_TINH_CHINH) / 1_000, color="C0", lw=2, label="tiền huấn luyện một lần + tinh chỉnh")
ax.axvline(diem_cat, color="C3", lw=0.8, ls=":")
ax.text(diem_cat + 0.3, 2, f"hoà vốn ≈ {diem_cat:.1f} tác vụ", fontsize=8, color="C3")
ax.plot([TAC_VU, TAC_VU], [TAC_VU * gio_moi_tac_vu / 1_000, (TIEN_HL + TAC_VU * GIO_TINH_CHINH) / 1_000], "o", color="C1")
ax.annotate("10 tác vụ: 10,000 so với 10,500", (TAC_VU, 10.25), textcoords="offset points", xytext=(-150, 30), fontsize=8,
            arrowprops={"arrowstyle": "-", "color": "0.5"})
ax.set_xlabel("số tác vụ hạ nguồn")
ax.set_ylabel("tổng nghìn giờ GPU")
ax.set_title("Khấu hao tiền huấn luyện, hình 7 của chương")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Ở đúng 10 tác vụ, tổng tính toán của mô hình nền tảng vẫn **cao hơn** một chút, vì 10 tác vụ nằm ngay dưới
điểm hoà vốn. Phần thắng chắc chắn ở đây là tiền nhãn, giảm 100 lần. Chương nói rõ cả hai con số là hệ quả của
giá trị kịch bản đã chọn: kết quả tuỳ chi phí tiền huấn luyện, số lần dùng lại và yêu cầu thích ứng của từng
tác vụ.

## 8. Ít phép toán hơn chưa chắc nhanh hơn

Chọn lọc dữ liệu không đứng một mình: mô hình học trên tập đã chọn còn đi qua nén và triển khai. Chương nhắc
một cái bẫy quen của nén mô hình: giảm 99 phần trăm số phép toán có thể giảm 0 phần trăm thời gian, vì phần
cứng nhân ma trận dày thưởng cho bố trí đều, còn ma trận thưa đòi truy cập bộ nhớ không đều. Đo trên CPU này:
nhân một ma trận 2,048 × 2,048 với 256 vectơ, ở dạng dày và dạng thưa CSR, với số phần tử khác 0 giảm dần.

In [ ]:
GIAM_99 = sach(99, nguon="99 phần trăm · 0 phần trăm · 2021") / 100
MAT_DO = [1.0, 0.5, 0.1, round(1 - GIAM_99, 2)]
g = torch.Generator().manual_seed(SEED)
W = torch.randn(2_048, 2_048, generator=g)
x = torch.randn(2_048, 256, generator=g)


def thoi_gian(ham, lap=5):
    ham()
    t0 = time.perf_counter()
    for _ in range(lap):
        ham()
    return (time.perf_counter() - t0) / lap


day = thoi_gian(lambda: W @ x)
do_tren_may("ma trận dày, đầy đủ", 1e3 * day, "ms")
ket_qua_thua = []
for md in MAT_DO:
    W_thua = (W * (torch.rand(W.shape, generator=g) < md)).to_sparse_csr()
    t = thoi_gian(lambda: W_thua @ x)
    ket_qua_thua.append((md, t))
    print(f"⏱ thưa CSR, {md:.0%} phần tử khác 0: {1e3 * t:6.1f} ms, phép toán còn {md:.0%},"
          f" thời gian còn {t / day:.0%} của dạng dày (đo trên máy này)")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.loglog([100 * md for md, _ in ket_qua_thua], [100 * t / day for _, t in ket_qua_thua], "o-", color="C3", lw=2,
          label="thưa CSR, đo trên máy này")
ax.loglog([100 * md for md in MAT_DO], [100 * md for md in MAT_DO], "--", color="0.4", lw=1.2,
          label="nếu thời gian tỉ lệ với phép toán")
ax.axhline(100, color="0.6", lw=0.8)
ax.text(1.1, 112, "dạng dày", fontsize=8, color="0.4")
ax.set_xticks([1, 10, 50, 100], ["1%", "10%", "50%", "100%"])
ax.set_yticks([1, 10, 100, 500], ["1%", "10%", "100%", "500%"])
ax.minorticks_off()
ax.set_xlabel("phần phép toán còn lại (phần tử khác 0)")
ax.set_ylabel("thời gian so với dạng dày")
ax.set_title("Phép toán và thời gian (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Ở mật độ 50 phần trăm, dạng thưa chậm hơn hẳn dạng dày dù làm một nửa số phép toán. Ngay cả ở 1 phần trăm,
thời gian giảm ít hơn nhiều so với số phép toán. Đó là lời của chương, theo Hoefler và cộng sự: tối ưu phải
nhắm vào điểm nghẽn thật của phần cứng, không chỉ một chỉ số trừu tượng.

Chọn lọc dữ liệu gặp đúng bẫy ấy ở quy mô cả hệ thống. Giảm 80 phần trăm tập dữ liệu chỉ cắt phần việc tỉ lệ
theo mẫu. Phần không đổi, như đánh giá sau mỗi epoch, nạp checkpoint hay chấm điểm chọn lọc, giờ chiếm phần
lớn hơn, và trở thành điểm nghẽn kế tiếp.

In [ ]:
GIAM_80 = sach(80, nguon="80 phần trăm") / 100
print(f"{'phần không đổi':>15} {'nhanh hơn khi bỏ 80% dữ liệu':>30} {'phần không đổi sau khi bỏ':>27}")
for co_dinh in (0.0, 0.05, 0.1, 0.25, 0.5):
    sau = co_dinh + (1 - co_dinh) * (1 - GIAM_80)
    print(f"{co_dinh:>15.0%} {1 / sau:>29.2f}× {co_dinh / sau:>26.0%}")

Chỉ khi không có phần cố định nào thì bỏ 80 phần trăm dữ liệu mới nhanh hơn 5 lần. Với một phần tư thời gian
là cố định, con số chỉ còn 2.5 lần, và phần cố định ấy giờ chiếm hơn nửa thời gian chạy. Vì vậy chương
dặn hãy profile hệ thống trước khi giảm dữ liệu mạnh tay, để biết điểm nghẽn nào đang được nhắm tới.

## Thử thêm

1. Ở phần 1, đổi MNIST thành FashionMNIST (`B`, `nhan_b`, `B_kt`, `nhan_b_kt`). Lật ngang còn làm hại không?
   Vì sao?
2. Ở phần 3, đặt `lat=False` và thêm `xoay=30` vào `bien_doi` trong `lap_lai`. Tăng cường mạnh hơn có làm
   "lặp trước tăng cường" gần ảnh mới hơn không?
3. Ở phần 5, đặt `gia_nhan=5` (giá nhãn chuyên gia của phép tính nhẩm 1.3, ở sổ tay 03) và `lan_chay=1`.
   Phần dữ liệu là bao nhiêu, và chọn lọc trước khi gán nhãn tiết kiệm được bao nhiêu?

## Tóm lại

* Tăng cường đúng bất biến của tác vụ thay được cho dữ liệu thật đắt; sai bất biến thì làm hại, nên phải kiểm
  định trên mô hình đích.
* Dữ liệu tổng hợp khác miền làm hại khi đã có vài trăm ảnh thật, và chỉ thêm chút ít khi ảnh thật cực hiếm;
  tỉ lệ pha trộn của bảng 9 là khung để đo, không phải công thức.
* Lặp lại dữ liệu chỉ có lợi khi R lớn hơn 1, và một batch lặp chỉ lấy lại một phần lợi ích của một batch mới,
  nên thời gian phép tính nhẩm hứa phải được đo lại tới cùng độ chính xác.
* Tỉ lệ dữ liệu trên tính toán, ROI của học chủ động và ROI khấu hao đều là hệ quả của giả định: giá nhãn, số
  lần chạy, chi phí mỗi vòng, số lần dùng lại. Hãy quét chúng trước khi tin một tỉ lệ.
* Ít phép toán hay ít mẫu hơn chưa chắc nhanh hơn: phần không đổi và điểm nghẽn phần cứng quyết định thời gian.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Chọn lọc dữ liệu — bớt việc trước khi việc bắt đầu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch09-data-selection/), dựng từ chương
[*Data Selection*](https://mlsysbook.ai/vol1/data_selection/data_selection.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.